[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Open-Medica/open-medical-skills/blob/main/oms-models/notebooks/03_train_embeddinggemma.ipynb)

# 03 - Fine-Tune EmbeddingGemma-300M

**OMS Custom Embedding Model Pipeline - Step 3 of 6**

This notebook fine-tunes [EmbeddingGemma-300M](https://huggingface.co/google/embeddinggemma) -- a 300M parameter
state-of-the-art embedding model from Google -- on the validated OMS training pairs.

**Why EmbeddingGemma?**
- 300M params -- small enough for on-device deployment
- 768-dim embeddings with Matryoshka support (768/512/256/128)
- Top performance in its size class (beats models 5x larger on MTEB)
- Built on Gemma 3 architecture with T5Gemma initialization

**Training approach:**
- Loss: CachedMultipleNegativesRankingLoss (memory-efficient contrastive)
- Data: (query, tool_description) pairs from Step 2
- Epochs: 3 with warmup
- Output: `oms-toolrag-gemma-v1`

**GPU requirements:** T4 (16GB) minimum. A100 recommended for faster training.

---

## 1. Setup & Dependencies

In [ ]:
!pip install -q sentence-transformers>=3.0.0 torch accelerate datasets pyyaml huggingface_hub

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
from datasets import DatasetDict, load_from_disk
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. GPU Check & Configuration

Detect the GPU type and configure training parameters accordingly.
EmbeddingGemma-300M is small enough for a T4, but batch sizes differ significantly.

In [ ]:
# GPU detection and auto-configuration
if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Training will be extremely slow.")
    print("In Colab: Runtime -> Change runtime type -> GPU")
    GPU_TYPE = 'cpu'
    BATCH_SIZE = 16
    MINI_BATCH_SIZE = 4
    FP16 = False
else:
    gpu_name = torch.cuda.get_device_name(0).lower()
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9

    if 'a100' in gpu_name or vram_gb >= 40:
        GPU_TYPE = 'A100'
        BATCH_SIZE = 256
        MINI_BATCH_SIZE = 32
        FP16 = True
    elif 'v100' in gpu_name or vram_gb >= 16:
        GPU_TYPE = 'V100/T4'
        BATCH_SIZE = 128
        MINI_BATCH_SIZE = 16
        FP16 = True
    else:
        GPU_TYPE = 'Other'
        BATCH_SIZE = 64
        MINI_BATCH_SIZE = 8
        FP16 = True

print(f"GPU type:        {GPU_TYPE}")
print(f"Batch size:      {BATCH_SIZE}")
print(f"Mini-batch size: {MINI_BATCH_SIZE}")
print(f"FP16:            {FP16}")

In [ ]:
# Mount Google Drive for checkpoints (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = '/content/drive/MyDrive/oms-models/checkpoints/oms-toolrag-gemma-v1'

OUTPUT_DIR = 'oms-toolrag-gemma-v1'
print(f"Model output directory: {OUTPUT_DIR}")

## 3. Load Training Configuration

Load the training config. If a config.yaml exists in the oms-models directory, use it.
Otherwise, use the defaults defined here.

In [ ]:
import yaml

# Default training config
config = {
    'model_name': 'google/embeddinggemma',
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': 3,
    'per_device_train_batch_size': BATCH_SIZE,
    'learning_rate': 2e-5,
    'warmup_ratio': 0.1,
    'fp16': FP16,
    'eval_steps': 100,
    'save_steps': 100,
    'mini_batch_size': MINI_BATCH_SIZE,
    'matryoshka_dims': [768, 512, 256, 128],
}

# Try to load from file
config_paths = [
    Path('config.yaml'),
    Path('../training/config.yaml'),
    Path('oms-models/training/config.yaml'),
]
for config_path in config_paths:
    if config_path.exists():
        with open(config_path) as f:
            file_config = yaml.safe_load(f)
        if file_config:
            config.update(file_config)
            print(f"Loaded config from: {config_path}")
        break
else:
    print("Using default training config.")

print(f"\nTraining config:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 4. Load Training Data

Load the train/val splits from Step 2 and convert to the format expected by SentenceTransformerTrainer.

In [ ]:
from datasets import Dataset

# Try loading from HuggingFace DatasetDict first
data_paths = [
    Path('data/processed/oms_training_pairs'),
    Path('../data/processed/oms_training_pairs'),
    Path('/content/drive/MyDrive/oms-models/data/processed/oms_training_pairs'),
]

dataset = None
for data_path in data_paths:
    if data_path.exists():
        dataset = load_from_disk(str(data_path))
        print(f"Loaded dataset from: {data_path}")
        break

if dataset is None:
    # Fall back to JSONL files
    for base_dir in [Path('data/processed'), Path('../data/processed')]:
        train_path = base_dir / 'train.jsonl'
        val_path = base_dir / 'val.jsonl'
        if train_path.exists():
            train_records = [json.loads(line) for line in open(train_path) if line.strip()]
            val_records = [json.loads(line) for line in open(val_path) if line.strip()]
            dataset = DatasetDict({
                'train': Dataset.from_list(train_records),
                'validation': Dataset.from_list(val_records),
            })
            print(f"Loaded from JSONL: {base_dir}")
            break

assert dataset is not None, "No training data found. Run notebooks 01 and 02 first."

print(f"\nDataset:")
print(dataset)

# The training data needs 'query' and 'tool_description' columns
# which map to anchor and positive in contrastive learning
print(f"\nColumns: {dataset['train'].column_names}")
print(f"Train examples: {len(dataset['train'])}")
print(f"Val examples: {len(dataset['validation'])}")

# Show a sample
sample = dataset['train'][0]
print(f"\nSample pair:")
print(f"  Query: {sample['query']}")
print(f"  Description: {sample['tool_description'][:150]}...")

In [ ]:
# Rename columns for sentence-transformers (anchor, positive)
train_ds = dataset['train'].rename_columns({'query': 'anchor', 'tool_description': 'positive'})
val_ds = dataset['validation'].rename_columns({'query': 'anchor', 'tool_description': 'positive'})

# Keep only the columns needed for training
train_ds = train_ds.select_columns(['anchor', 'positive'])
val_ds = val_ds.select_columns(['anchor', 'positive'])

print(f"Training dataset: {len(train_ds)} pairs")
print(f"Validation dataset: {len(val_ds)} pairs")
print(f"Columns: {train_ds.column_names}")

## 5. Load Base Model

Load the pre-trained EmbeddingGemma-300M model. This requires accepting the Gemma license on HuggingFace.

In [ ]:
# Login to HuggingFace (required for gated model)
# Option 1: Set HF_TOKEN env var
# Option 2: Run huggingface-cli login
# Option 3: Use Colab secrets

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("Loaded HF_TOKEN from Colab Secrets")
except (ImportError, Exception):
    if not os.getenv('HF_TOKEN'):
        print("HF_TOKEN not set. You may need to login:")
        print("  !huggingface-cli login --token YOUR_TOKEN")

# Try loading from local path first (if model is pre-downloaded)
local_model_paths = [
    'models/embeddinggemma-300m',
    '../models/embeddinggemma-300m',
    'oms-models/models/embeddinggemma-300m',
]

model_path = config['model_name']  # Default: HuggingFace hub
for local_path in local_model_paths:
    if Path(local_path).exists():
        model_path = local_path
        print(f"Using local model: {local_path}")
        break

print(f"Loading model from: {model_path}")

In [ ]:
model = SentenceTransformer(model_path, trust_remote_code=True)

print(f"Model loaded successfully.")
print(f"\nModel architecture:")
print(model)
print(f"\nEmbedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params / 1e6:.1f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.1f}M")

## 6. Configure Training

Set up the loss function, training arguments, and evaluator.

We use **CachedMultipleNegativesRankingLoss** which is memory-efficient -- it computes the full
contrastive loss using gradient caching, allowing large effective batch sizes even on smaller GPUs.

In [ ]:
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import (
    BatchSamplers,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.trainer import SentenceTransformerTrainer

# Loss function with gradient caching
loss = CachedMultipleNegativesRankingLoss(
    model,
    mini_batch_size=config['mini_batch_size'],
)

# Training arguments
args = SentenceTransformerTrainingArguments(
    output_dir=config['output_dir'],
    num_train_epochs=config['num_train_epochs'],
    per_device_train_batch_size=config['per_device_train_batch_size'],
    per_device_eval_batch_size=config['per_device_train_batch_size'],
    learning_rate=config['learning_rate'],
    warmup_ratio=config['warmup_ratio'],
    fp16=config['fp16'],
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy='steps',
    eval_steps=config['eval_steps'],
    save_strategy='steps',
    save_steps=config['save_steps'],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    report_to='none',  # Set to 'wandb' if using Weights & Biases
)

print("Training configuration:")
print(f"  Epochs: {args.num_train_epochs}")
print(f"  Batch size: {args.per_device_train_batch_size}")
print(f"  Learning rate: {args.learning_rate}")
print(f"  Warmup ratio: {args.warmup_ratio}")
print(f"  FP16: {args.fp16}")
print(f"  Eval steps: {args.eval_steps}")
print(f"  Save steps: {args.save_steps}")

# Estimated training steps
steps_per_epoch = len(train_ds) // args.per_device_train_batch_size
total_steps = steps_per_epoch * args.num_train_epochs
print(f"\nEstimated steps: {total_steps} ({steps_per_epoch}/epoch)")

## 7. Train

Run the SentenceTransformerTrainer. Training time depends on GPU:
- **T4**: ~45 minutes for 3 epochs on 25K pairs
- **A100**: ~10 minutes for 3 epochs on 25K pairs

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    loss=loss,
)

print("Starting training...")
print(f"Monitor loss in the output below.\n")

train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training time: {train_result.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
# Plot training loss curve (if training logs are available)
try:
    import matplotlib.pyplot as plt

    logs = trainer.state.log_history
    train_losses = [(log['step'], log['loss']) for log in logs if 'loss' in log]
    eval_losses = [(log['step'], log['eval_loss']) for log in logs if 'eval_loss' in log]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))

    if train_losses:
        steps, losses = zip(*train_losses)
        ax.plot(steps, losses, label='Train Loss', alpha=0.7)

    if eval_losses:
        steps, losses = zip(*eval_losses)
        ax.plot(steps, losses, label='Eval Loss', marker='o', markersize=4)

    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('EmbeddingGemma Fine-Tuning Loss Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not plot loss curve: {e}")

## 8. Quick Evaluation

Run a quick sanity check: encode 10 test queries and all tool descriptions, compute cosine similarity,
and show the top-5 retrieved tools for each query.

In [ ]:
from sentence_transformers.util import cos_sim

# Load test data
test_ds = None
if dataset is not None and 'test' in dataset:
    test_ds = dataset['test']
else:
    for path in [Path('data/processed/test.jsonl'), Path('../data/processed/test.jsonl')]:
        if path.exists():
            test_records = [json.loads(line) for line in open(path) if line.strip()]
            from datasets import Dataset as HFDataset
            test_ds = HFDataset.from_list(test_records)
            break

if test_ds is None:
    print("No test dataset found. Using validation set for quick eval.")
    test_ds = dataset['validation']

# Sample 10 test queries
import random
random.seed(42)
test_indices = random.sample(range(len(test_ds)), min(10, len(test_ds)))
test_queries = [test_ds[i]['query'] for i in test_indices]
test_ground_truth = [test_ds[i]['tool_name'] for i in test_indices]

# Get unique tool descriptions as corpus
all_records_for_corpus = list(dataset['train']) + list(dataset['validation'])
if 'test' in dataset:
    all_records_for_corpus += list(dataset['test'])

corpus = {}
for r in all_records_for_corpus:
    tool_name = r['tool_name']
    if tool_name not in corpus:
        corpus[tool_name] = r['tool_description']

corpus_names = list(corpus.keys())
corpus_descs = list(corpus.values())

print(f"Test queries: {len(test_queries)}")
print(f"Corpus size: {len(corpus_names)} unique tools")

In [ ]:
# Encode queries and corpus
query_embeddings = model.encode(test_queries, convert_to_tensor=True, show_progress_bar=True)
corpus_embeddings = model.encode(corpus_descs, convert_to_tensor=True, show_progress_bar=True, batch_size=64)

# Compute similarities
similarities = cos_sim(query_embeddings, corpus_embeddings)

# Show results
print(f"\n{'='*80}")
print(f"RETRIEVAL RESULTS (Fine-Tuned EmbeddingGemma)")
print(f"{'='*80}")

hits = 0
for i, (query, gt_tool) in enumerate(zip(test_queries, test_ground_truth)):
    scores, indices = similarities[i].topk(5)
    top5_names = [corpus_names[idx] for idx in indices]

    is_hit = gt_tool in top5_names
    if is_hit:
        hits += 1

    hit_marker = 'HIT' if is_hit else 'MISS'
    print(f"\n[{hit_marker}] Query: {query[:80]}")
    print(f"  Ground truth: {gt_tool}")
    print(f"  Top-5:")
    for rank, (name, score) in enumerate(zip(top5_names, scores), 1):
        marker = ' <<' if name == gt_tool else ''
        print(f"    {rank}. {name} (sim={score:.4f}){marker}")

print(f"\nRecall@5: {hits}/{len(test_queries)} ({100*hits/len(test_queries):.1f}%)")

## 9. Save Model

Save the fine-tuned model locally and optionally push to HuggingFace Hub.

In [ ]:
# Save locally
model.save(config['output_dir'])
print(f"Model saved to: {config['output_dir']}")

# List saved files
saved_files = list(Path(config['output_dir']).glob('*'))
for f in sorted(saved_files):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name} ({size_mb:.1f} MB)" if size_mb > 0.1 else f"  {f.name}")

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and set your HF organization/username

# HF_REPO = 'intelmedica/oms-toolrag-gemma-v1'
# model.push_to_hub(
#     HF_REPO,
#     private=True,
#     commit_message='Fine-tuned EmbeddingGemma-300M on OMS medical tool retrieval data',
# )
# print(f"Model pushed to: https://huggingface.co/{HF_REPO}")

## 10. Matryoshka Dimension Test

EmbeddingGemma supports Matryoshka Representation Learning (MRL), meaning embeddings can be
truncated to smaller dimensions (768 -> 512 -> 256 -> 128) with minimal quality loss.

Test retrieval quality at each dimension to determine the optimal size/quality tradeoff.

In [ ]:
import torch.nn.functional as F

DIMS_TO_TEST = config.get('matryoshka_dims', [768, 512, 256, 128])

print(f"Testing Matryoshka dimensions: {DIMS_TO_TEST}")
print(f"{'='*70}")

results = []

for dim in DIMS_TO_TEST:
    # Truncate embeddings to target dimension and re-normalize
    q_trunc = F.normalize(query_embeddings[:, :dim], p=2, dim=1)
    c_trunc = F.normalize(corpus_embeddings[:, :dim], p=2, dim=1)

    sims = cos_sim(q_trunc, c_trunc)

    # Compute Recall@1 and Recall@5
    recall_at_1 = 0
    recall_at_5 = 0
    for i, gt_tool in enumerate(test_ground_truth):
        top1_idx = sims[i].argmax().item()
        top5_indices = sims[i].topk(5).indices.tolist()

        if corpus_names[top1_idx] == gt_tool:
            recall_at_1 += 1
        if gt_tool in [corpus_names[idx] for idx in top5_indices]:
            recall_at_5 += 1

    r1 = 100 * recall_at_1 / len(test_queries)
    r5 = 100 * recall_at_5 / len(test_queries)
    results.append({'dim': dim, 'recall@1': r1, 'recall@5': r5})

    print(f"  dim={dim:>4}:  Recall@1={r1:5.1f}%  Recall@5={r5:5.1f}%")

print(f"\nRecommendation:")
for r in results:
    if r['recall@5'] >= results[0]['recall@5'] * 0.95:  # Within 5% of full dim
        best_compact = r['dim']
print(f"  Smallest dim within 5%% of full quality: {best_compact}")
print(f"  Memory savings: {(1 - best_compact/768)*100:.0f}% vs full 768-dim")

---

## Next Steps

- **04_train_gte_qwen.ipynb** -- Fine-tune GTE-Qwen2-1.5B as a secondary (higher-quality) model
- **05_evaluate_models.ipynb** -- Full benchmark comparison of all models

**Files produced by this notebook:**
- `oms-toolrag-gemma-v1/` -- Fine-tuned model weights and config
- Checkpoint directories under `oms-toolrag-gemma-v1/checkpoint-*/`